---

# SETUP & IMPORTS

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import statistics
import json
import os

# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Imports successful")
print(f"📅 Today: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---

# 1️⃣ PERPLEXITY: INSTITUTIONAL VS RETAIL VOLUME DETECTOR

**Problem:** How do we tell if institutional or retail money is driving a move?  
**Solution:** Analyze block trades, VWAP relationships, volume patterns, order types.

In [ ]:
class InstitutionalVsRetailDetector:
    """
    PERPLEXITY'S SOLUTION - Distinguish institutional accumulation from retail chasing.
    
    Based on research: Institutions trade to minimize market impact (VWAP-based),
    while retail chases price with market orders.
    """
    
    def __init__(self):
        self.vwap_lookback = 60  # minutes
        self.block_threshold = 50000  # shares (institutional block size)
    
    def analyze_volume_signature(
        self, 
        time_and_sales: List[dict], 
        vwap: float, 
        time_window: Tuple[datetime, datetime]
    ) -> Tuple[str, float, dict]:
        """
        Analyze who is driving the volume.
        
        Returns: (trader_type, confidence, evidence)
        trader_type: 'INSTITUTIONAL' | 'RETAIL' | 'MIXED' | 'UNKNOWN'
        """
        
        # Filter trades to time window
        window_trades = [
            t for t in time_and_sales
            if time_window[0] <= t['time'] <= time_window[1]
        ]
        
        if not window_trades:
            return 'UNKNOWN', 0, {}
        
        # Feature 1: Block Trade Analysis
        blocks = [t for t in window_trades if t['shares'] >= self.block_threshold]
        block_score = self._score_blocks(blocks, vwap)
        
        # Feature 2: Price vs VWAP Relationship
        vwap_score = self._score_vwap_relationship(window_trades, vwap)
        
        # Feature 3: Volume Accumulation Pattern
        volume_score = self._score_volume_pattern(window_trades)
        
        # Feature 4: Order Type Detection
        order_score = self._score_order_types(window_trades)
        
        # Composite score
        institutional_confidence = (
            0.40 * block_score +
            0.30 * vwap_score +
            0.20 * volume_score +
            0.10 * order_score
        )
        
        evidence = {
            'total_blocks': len(blocks),
            'blocks_below_vwap': sum(1 for b in blocks if b['price'] < vwap),
            'total_volume': sum(t['shares'] for t in window_trades),
            'block_score': round(block_score, 2),
            'vwap_score': round(vwap_score, 2),
            'volume_pattern_score': round(volume_score, 2),
            'order_type_score': round(order_score, 2),
        }
        
        if institutional_confidence > 0.70:
            return 'INSTITUTIONAL', institutional_confidence, evidence
        elif institutional_confidence < 0.30:
            return 'RETAIL', 1 - institutional_confidence, evidence
        else:
            return 'MIXED', institutional_confidence, evidence
    
    def _score_blocks(self, blocks: List[dict], vwap: float) -> float:
        """Institutional: Multiple blocks trading at/below VWAP"""
        if len(blocks) < 2:
            return 0.2  # No blocks = retail
        
        blocks_at_vwap = sum(
            1 for b in blocks
            if abs(b['price'] - vwap) / vwap < 0.005
        )
        
        blocks_below_vwap = sum(1 for b in blocks if b['price'] < vwap)
        
        vwap_adherence = (blocks_at_vwap + blocks_below_vwap * 0.5) / len(blocks)
        block_consistency = min(len(blocks) / 5, 1.0)
        
        return vwap_adherence * 0.7 + block_consistency * 0.3
    
    def _score_vwap_relationship(self, trades: List[dict], vwap: float) -> float:
        """Calculate % of volume traded below VWAP"""
        total_volume = sum(t['shares'] for t in trades)
        
        if total_volume == 0:
            return 0.5
        
        volume_below_vwap = sum(
            t['shares'] for t in trades if t['price'] < vwap
        )
        
        pct_below = volume_below_vwap / total_volume
        
        # Institutional: 45-60% below (buying efficiently)
        # Retail: <30% below (chasing price)
        if pct_below > 0.60:
            return 1.0
        elif pct_below > 0.45:
            return 0.75
        elif pct_below > 0.30:
            return 0.40
        else:
            return 0.1
    
    def _score_volume_pattern(self, trades: List[dict]) -> float:
        """Institutional: Consistent volume. Retail: Spiky volume"""
        minute_volumes = {}
        for t in trades:
            minute = t['time'].replace(second=0, microsecond=0)
            minute_volumes[minute] = minute_volumes.get(minute, 0) + t['shares']
        
        volumes = list(minute_volumes.values())
        
        if len(volumes) < 2:
            return 0.5
        
        mean_vol = statistics.mean(volumes)
        std_vol = statistics.stdev(volumes)
        cv = std_vol / mean_vol if mean_vol > 0 else 1.0
        
        # Low CV = consistent = institutional
        if cv < 0.5:
            return 1.0
        elif cv < 1.0:
            return 0.6
        else:
            return 0.2
    
    def _score_order_types(self, trades: List[dict]) -> float:
        """Infer limit vs market orders from price clustering"""
        price_clusters = {}
        for t in trades:
            price = round(t['price'], 2)
            price_clusters[price] = price_clusters.get(price, 0) + t['shares']
        
        if len(price_clusters) == 0:
            return 0.5
        
        sorted_prices = sorted(price_clusters.items(), key=lambda x: x[1], reverse=True)
        
        top_3_volume = sum(p[1] for p in sorted_prices[:3])
        total_volume = sum(t['shares'] for t in trades)
        
        pct_top_3 = top_3_volume / total_volume if total_volume > 0 else 0
        
        # High concentration = limit orders = institutional
        if pct_top_3 > 0.70:
            return 0.85
        elif pct_top_3 > 0.50:
            return 0.60
        else:
            return 0.20

print("✅ Perplexity's InstitutionalVsRetailDetector loaded")

---

# 2️⃣ DEEPSEEK: EVOLVED DIP-BUYING SYSTEM

**Problem:** Distinguish buyable dips from falling knives  
**Solution:** 5-filter system + expected value calculator + PDT simulation

In [ ]:
def is_buyable_dip_v2_deepseek(
    stock_data: dict,
    spy_data: dict,
    sector_data: dict,
    news_data: dict,
    vix_data: dict,
    current_time: datetime
) -> Tuple[bool, str]:
    """
    DEEPSEEK'S COMPLETE 5-FILTER DIP SYSTEM
    
    Returns: (is_buyable, reason)
    """
    
    # BASE CONDITIONS: Strong stock in strong market
    base_ok = (
        stock_data['close'] > stock_data['ma_200'] and
        spy_data['close'] > spy_data['ma_50'] and
        -8.0 <= stock_data['pct_change'] <= -3.5
    )
    
    if not base_ok:
        return False, "Base conditions not met"
    
    failures = []
    
    # FILTER 1: SECTOR RELATIVE STRENGTH
    sector_ok = sector_data['pct_change'] > stock_data['pct_change']
    if not sector_ok:
        failures.append("sector_weakness")
    
    # FILTER 2: VOLUME PATTERN (MOST CRITICAL)
    volume_ok = (
        stock_data['volume'] < (stock_data['avg_volume_20'] * 0.85) and
        stock_data['volume_prev'] > (stock_data['avg_volume_20'] * 1.2) and
        stock_data['close_prev'] > stock_data['open_prev']
    )
    if not volume_ok:
        failures.append("bad_volume_pattern")
    
    # FILTER 3: NEWS SENTIMENT
    blacklist_keywords = ['fraud', 'investigation', 'sec', 'downgrade', 'cut', 'miss', 'warning', 'sues']
    headline = news_data.get('headline', '').lower()
    body = news_data.get('body', '').lower()
    news_ok = not any(kw in headline or kw in body for kw in blacklist_keywords)
    if not news_ok:
        failures.append("bad_news")
    
    # FILTER 4: VIX - ABSOLUTE & RELATIVE SPIKE
    vix_ok = (
        vix_data['current'] < 22 and
        vix_data['current'] < (vix_data['ma_20'] * 1.15)
    )
    if not vix_ok:
        failures.append("vix_spike")
    
    # FILTER 5: TIME OF DAY
    hour_min = current_time.hour + current_time.minute / 60
    time_ok = 10.5 <= hour_min <= 14.5
    if not time_ok:
        failures.append("bad_entry_time")
    
    # DECISION
    all_ok = sector_ok and volume_ok and news_ok and vix_ok and time_ok
    
    if all_ok:
        return True, "All filters passed - BUYABLE DIP"
    else:
        return False, f"Failed filters: {', '.join(failures)}"

print("✅ DeepSeek's is_buyable_dip_v2_deepseek loaded")

In [ ]:
def calculate_expected_value_and_position(
    historical_trades: pd.DataFrame,
    capital: float,
    max_kelly_fraction: float = 0.25
) -> Tuple[float, float, str]:
    """
    DEEPSEEK'S EXPECTED VALUE CALCULATOR
    Uses Kelly Criterion for position sizing.
    
    Returns: (expected_value, position_size, explanation)
    """
    
    if len(historical_trades) < 10:
        return 0, 0, "Insufficient data: need at least 10 historical trades"
    
    wins = historical_trades[historical_trades['gain'] > 0]
    losses = historical_trades[historical_trades['loss'] < 0]
    
    # Win rate
    W = len(wins) / len(historical_trades)
    
    # Average win/loss (median to reduce outliers)
    avg_win = wins['gain'].median()
    avg_loss = abs(losses['loss'].median())
    
    if avg_loss == 0:
        return 0, 0, "Avg loss is zero, cannot calculate"
    
    # Win/Loss ratio
    R = avg_win / avg_loss
    
    # Kelly Criterion
    K = W - ((1 - W) / R)
    
    # Conservative: Half-Kelly
    conservative_K = K / 2
    position_fraction = min(conservative_K, max_kelly_fraction)
    position_fraction = max(position_fraction, 0)
    
    position_size = capital * position_fraction
    expected_value = (W * avg_win) - ((1 - W) * avg_loss)
    
    explanation = (
        f"Win Rate: {W:.2%}, Avg Win/Loss: {avg_win:.2f}/{avg_loss:.2f}, "
        f"Kelly%: {K:.2%}, Use%: {position_fraction:.2%}"
    )
    
    return expected_value, position_size, explanation

print("✅ DeepSeek's calculate_expected_value_and_position loaded")

In [ ]:
def simulate_day_trade_usage(
    num_simulations: int = 10000,
    positions_per_week: int = 3,
    weeks: int = 52,
    stop_hit_rate: float = 0.25,
    emergency_rate: float = 0.05
) -> dict:
    """
    DEEPSEEK'S PDT SIMULATION
    Monte Carlo simulation of day trade usage under PDT rules.
    
    Returns: Statistics on how often we get stuck
    """
    
    results = []
    day_trades_per_week = []
    
    for _ in range(num_simulations):
        day_trades_used = 0
        stuck_positions = 0
        
        for week in range(weeks):
            # Reset every 5 trading days (simplified)
            if week % 1 == 0:
                day_trades_used = max(0, day_trades_used - 3)
            
            weekly_trades = np.random.binomial(positions_per_week, 1, positions_per_week)
            
            for trade in weekly_trades:
                # Emergency exit (Black Swan)
                if np.random.rand() < emergency_rate:
                    if day_trades_used < 3:
                        day_trades_used += 1
                    else:
                        stuck_positions += 1
                
                # Strategic stop loss hit
                elif np.random.rand() < stop_hit_rate:
                    if day_trades_used < 3:
                        day_trades_used += 1
                    else:
                        stuck_positions += 1
        
        day_trades_per_week.append(day_trades_used)
        results.append(stuck_positions)
    
    avg_stuck = np.mean(results)
    pct_weeks_with_stuck = np.mean(np.array(results) > 0) * 100
    avg_trades_used = np.mean(day_trades_per_week)
    
    print(f"=== PDT SIMULATION RESULTS ({num_simulations} runs) ===")
    print(f"Avg. Stuck Positions per Trader: {avg_stuck:.2f}")
    print(f"% of Simulations with ANY Stuck Position: {pct_weeks_with_stuck:.1f}%")
    print(f"Avg. Day Trades Used per Week: {avg_trades_used:.2f}/3")
    print(f"\nInterpretation: With a {stop_hit_rate:.0%} stop rate and {emergency_rate:.0%} emergency rate,")
    print(f"you'll run out of day trades in ~{pct_weeks_with_stuck:.1f}% of weeks.")
    
    return {
        'avg_stuck': avg_stuck,
        'pct_with_stuck': pct_weeks_with_stuck,
        'simulation_results': results
    }

print("✅ DeepSeek's simulate_day_trade_usage loaded")

---

# 3️⃣ CLAUDE: CONFIDENCE FUNCTION & POSITION SIZING

**Problem:** Multi-day wave prediction with capital allocation  
**Solution:** 6-component confidence scorer + 3-tier position sizer + gap exhaustion

In [ ]:
# Claude's complete implementation in next cell due to length
# This is the FULL working code from the AI Council response